## Bronze Layer :
1. Load Raw CSV
2. Skip Invalid Hedaer Row
3. Validate Column Structure

In [1]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\shivn\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd     # For data manipulation
import numpy as np      # For handling missing values (NaN)
import os               # For navigating your folders
import sys              # To check system-level info if needed

In [3]:
# 1. Define the file path (The 'Bronze' path)
file_path = r"C:\Users\shivn\NASA-ASRS-Pipelines\data\bronze\ASRS_DBOnline.csv"

# 2. Load the messy data
df_bronze = pd.read_csv(file_path, low_memory=False)

#We use low_memory=False because raw aviation data often has mixed numbers and text in the same column, which confuses Python.

print("Data Loaded Successfully!","Shape:",df_bronze.shape)

Data Loaded Successfully! Shape: (4502, 125)


In [4]:
## Architecture Decision: Why 125 Columns, Not 250

# OBSERVATION: df_bronze.shape = (4502, 125)
# QUESTION: The NASA ASRS data dictionary lists ~250 conceptual fields.
#           Why does pandas show only 125?

# ROOT CAUSE: The NASA ASRS CSV uses a MIRRORED SCHEMA design.
# The file has 125 physical columns but 48 column names are duplicated.
# This is because every incident can involve TWO aircraft and TWO reporters:
#
#   Aircraft 1  → 35 columns  (primary aircraft)
#   Aircraft 2  → 35 columns  (SAME 35 column names, different context)
#   Person 1    → 11 columns  (primary reporter)
#   Person 2    → 11 columns  (SAME 11 column names, different context)
#   Report 1    →  3 columns  (Narrative, Callback, Synopsis)
#   Report 2    →  2 columns  (Narrative, Callback — second reporter)
#
# Total unique column NAMES: 77
# Total physical columns:   125
# Duplicate names:           48
#
# PANDAS BEHAVIOUR: pd.read_csv() with default header=0 reads Row 1 as
# column names. When it encounters a duplicate name, the later column
# silently overwrites the earlier one in the DataFrame index.
# Both physical columns ARE loaded — but Aircraft 1 data is shadowed
# by Aircraft 2 wherever names collide.
#
# CORRECT WAY TO READ (preserving both Aircraft 1 and Aircraft 2):

In [5]:
# Read with BOTH header rows to preserve category context
df_bronze_full = pd.read_csv(
    file_path,
    low_memory=False,
    header=[0, 1]   # Row 0 = category, Row 1 = column name
)
# Now columns become MultiIndex tuples: ('Aircraft 1', 'Make Model Name')
# vs ('Aircraft 2', 'Make Model Name') — no data loss
print("Full shape with MultiIndex:", df_bronze_full.shape)

# Access Aircraft 1 data cleanly
ac1_model = df_bronze_full[('Aircraft 1', 'Make Model Name')]

# Access Aircraft 2 data cleanly  
ac2_model = df_bronze_full[('Aircraft 2', 'Make Model Name')]

Full shape with MultiIndex: (4501, 125)


Code 1 answers the question — did my data arrive safely? It is your ingestion confirmation. Simple, fast, production standard.

Code 2 answers the question — can I trust what I am extracting? It is your data archaeology tool. When your Silver layer needs to pull Make Model Name for the primary aircraft only — you use Code 2 to ensure you are pulling Aircraft 1 and not Aircraft 2 accidentally.

The NASA ASRS schema mirrors itself for multi-aircraft incidents — Aircraft 1 and Aircraft 2 share identical column names. Pandas loaded all 125 physical columns correctly, but 48 duplicate names caused Aircraft 1 data to be shadowed by Aircraft 2. The Bronze layer preserves the raw file. The Silver layer resolves this using positional multi-index extraction with documented architectural reasoning.

In [6]:
# ── BRONZE LAYER COMPLETE ──────────────────────────────────────────────────
# Raw file: data/bronze/ASRS_DBOnline.csv
# Rows: 4502 (includes 1 ghost row — category header leak)
# Columns: 125 (48 duplicate names due to mirrored Aircraft1/Aircraft2 schema)
# Status: LOCKED. Never modify. Source of truth for all re-processing.
# Next: Silver layer transformation begins in src/silver_transform.py

print("Bronze Layer — LOCKED")
print(f"   Source file : data/bronze/ASRS_DBOnline.csv")
print(f"   Raw shape   : (4502, 125)")
print(f"   Ghost rows  : 1 (to be removed in Silver)")
print(f"   Dual schema : Aircraft 1 / Aircraft 2 mirrored (48 duplicate names)")
print(f"   Action      : Bronze is read-only.")

Bronze Layer — LOCKED
   Source file : data/bronze/ASRS_DBOnline.csv
   Raw shape   : (4502, 125)
   Ghost rows  : 1 (to be removed in Silver)
   Dual schema : Aircraft 1 / Aircraft 2 mirrored (48 duplicate names)
   Action      : Bronze is read-only.
